# 00 FGCR Common Core

This file serves as the shared code base for `01_Known_Parameter_Inference.ipynb`、`02_Finite_Sample_Inference.ipynb`, and `03_Do_Intervention_Inference.ipynb`.
**Scope of Responsibilities**
- Responsible for: loading the frozen protocol and enforcing hard assertions; generic discrete DAG/FGCR operations; exact variable elimination and an independent enumeration oracle; the shared Dirichlet parameter bank; generic child-slicing interventions; and common hashing, Excel-output, and environment-recording utilities.
- Single source of network facts:`data/alarm_frozen_protocol.json`. If the network, any of the four topological orders, or the intervention graph is modified, this file must raise an error immediately rather than continue silently.

## 1. Dependencies, Directories, and Frozen Protocol

In [1]:
# Allow type annotations to reference classes defined later, while preserving compatibility with dynamic loading in Notebooks 01–03.
from __future__ import annotations

# Standard-library modules for hashing, serialization, platform recording, and combinatorial enumeration.
import hashlib
import json
import math
import os
import platform
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import combinations, product
from pathlib import Path
from typing import Iterable, Mapping, MutableMapping, Sequence

# NumPy handles multi-state CPT tensors and exact probability computation; Pandas/OpenPyXL handle audit tables.
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

# Unified type aliases explicitly distinguish nodes, edges, CPT mappings, and parent-axis mappings.
Node = str
Edge = tuple[Node, Node]
CPTMap = dict[Node, np.ndarray]
ParentOrder = dict[Node, tuple[Node, ...]]

# Anchor all relative paths to the current project directory of the notebook so that Notebooks 01–03 can share the same resources.
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
RESULT_DIR = PROJECT_ROOT / "result"
#FIGURE_DIR = PROJECT_ROOT / "figure"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
COMMON_NOTEBOOK = PROJECT_ROOT / "00_FGCR_Common_Core.ipynb"
PROTOCOL_PATH = DATA_DIR / "alarm_frozen_protocol.json"
#PARAMETER_BANK_PATH = DATA_DIR / "alarm_parameter_sets.xlsx"

for _directory in (DATA_DIR, RESULT_DIR, CHECKPOINT_DIR):
    _directory.mkdir(parents=True, exist_ok=True)

if not PROTOCOL_PATH.exists():
    raise FileNotFoundError(
        "Missing data/alarm_frozen_protocol.json. Copy the final frozen machine "
        "protocol next to this notebook before loading 00."
    )
# The machine-readable protocol is the single source of truth for the network structure, states, four orders, and intervention definition.
FROZEN_PROTOCOL = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
if FROZEN_PROTOCOL.get("protocol_status") != "frozen_after_pilot":
    raise RuntimeError("The ALARM protocol is not in frozen_after_pilot state")

#  The parameter-bank file and its content hash are specified directly by the frozen protocol.
PARAMETER_BANK_PROTOCOL = FROZEN_PROTOCOL["parameter_bank"]
PARAMETER_BANK_PATH = (
    PROJECT_ROOT / PARAMETER_BANK_PROTOCOL["relative_path"]
)
FROZEN_PARAMETER_BANK_SHA256 = str(
    PARAMETER_BANK_PROTOCOL["content_sha256"]
)
N_PARAMETER_SETS = int(
    PARAMETER_BANK_PROTOCOL["n_parameter_sets"]
)
#  The state order and PARENT_ORDER jointly determine the meaning of each CPT axis; they must not be reordered independently.
NETWORK = FROZEN_PROTOCOL["network"]

# node_order determines which node receives each segment of the random stream, so it is part of the parameter-generation protocol.
# First verify that it contains exactly the same node set as states/parents, then reconstruct both ordered mappings according to node_order.
# This prevents external reordering of JSON object keys from changing the random CPT associated with any parameter_id.
# The explicit node_order determines both RNG allocation and all shared mapping orders; dependence on JSON key order is prohibited.
NODE_ORDER = tuple(NETWORK["node_order"])
if (
    len(NODE_ORDER) != len(set(NODE_ORDER))
    or set(NODE_ORDER) != set(NETWORK["states"])
    or set(NODE_ORDER) != set(NETWORK["parents"])
):
    raise AssertionError("network.node_order must list every ALARM node exactly once")
STATE_ORDER = {
    node: tuple(NETWORK["states"][node])
    for node in NODE_ORDER
}
PARENT_ORDER = {
    node: tuple(NETWORK["parents"][node])
    for node in NODE_ORDER
}
STATE_TO_INDEX = {
    node: {state: index for index, state in enumerate(levels)}
    for node, levels in STATE_ORDER.items()
}
INDEX_TO_STATE = {
    node: {index: state for index, state in enumerate(levels)}
    for node, levels in STATE_ORDER.items()
}
CARDINALITY = {node: len(levels) for node, levels in STATE_ORDER.items()}
ALARM_EDGES = frozenset(
    (parent, node)
    for node, parents in PARENT_ORDER.items()
    for parent in parents
)

#  All experiment constants below are read directly from the frozen protocol and must not be duplicated manually in Notebooks 01–03.
CONDITION = dict(FROZEN_PROTOCOL["condition"])
CONDITION_NODES = tuple(CONDITION)
TARGET = tuple(FROZEN_PROTOCOL["target"])
TARGET_EVENT = dict(FROZEN_PROTOCOL["target_event"])
INTERVENTION = dict(FROZEN_PROTOCOL["intervention"]["assignment"])
EXPECTED_INTERVENTION_CHILDREN = {
    node: tuple(children)
    for node, children in FROZEN_PROTOCOL["intervention"]["expected_children"].items()
}
BASE_SEED = int(FROZEN_PROTOCOL["base_seed"])
ISS = float(FROZEN_PROTOCOL["iss"])
#N_PARAMETER_SETS = 100
# Schema version 2 means that the network identity explicitly includes node_order; older parameter banks must not be mixed into the current frozen experiments.
ALARM_SCHEMA_VERSION = 2

# Store the four observational orders and their expected H_R structures for parent-set-level regression checks at load time.
ORDER_RECORDS = tuple(FROZEN_PROTOCOL["orders"])
BETA_ORDERS = {
    f"order_{record['id']}": tuple(record["beta"])
    for record in ORDER_RECORDS
}
EXPECTED_HR_PARENTS = {
    f"order_{record['id']}": {
        node: tuple(parents) for node, parents in record["hr_parents"].items()
    }
    for record in ORDER_RECORDS
}
#  The do-orders are frozen orders obtained by filtering the observational beta after removing the intervention nodes.
DO_ORDER_RECORDS = tuple(FROZEN_PROTOCOL["intervention"]["orders"])
DO_BETA_ORDERS = {
    f"order_{record['id']}": tuple(record["beta_do"])
    for record in DO_ORDER_RECORDS
}
EXPECTED_DO_HR_PARENTS = {
    f"order_{record['id']}": {
        node: tuple(parents) for node, parents in record["hr_do_parents"].items()
    }
    for record in DO_ORDER_RECORDS
}

#quick is for debugging only, pilot reproduces the small-scale timing setup, and formal corresponds to the full formal experiment.
EXPERIMENT_SETTINGS = {
    "quick": {
        "parameter_ids": tuple(range(3)),
        "sample_sizes": (100, 1000),
        "repetitions": 2,
    },
    "pilot": {
        "parameter_ids": tuple(FROZEN_PROTOCOL["pilot"]["parameter_ids"]),
        "sample_sizes": tuple(FROZEN_PROTOCOL["pilot"]["sample_sizes"]),
        "repetitions": int(FROZEN_PROTOCOL["pilot"]["repetitions"]),
    },
    "formal": {
        "parameter_ids": tuple(range(N_PARAMETER_SETS)),
        "sample_sizes": (100, 500, 1000, 2500, 5000, 7500, 10000),
        "repetitions": 100,
    },
}

# Encode an object as stable JSON bytes for content-identity calculations across repeated runs.
def canonical_json_bytes(value: object) -> bytes:
    """Stable JSON encoding consistent with the frozen script."""
    return json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")

# Compute a file SHA-256 incrementally to avoid unnecessary memory usage when hashing large workbooks.
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Recompute the semantic hash of the explicit node order, ALARM structure, states, and source to detect protocol drift.
def frozen_network_semantic_sha256() -> str:
    payload = {
        "node_order": NODE_ORDER,
        "states": NETWORK["states"],
        "parents": NETWORK["parents"],
        "source": "bnlearn ALARM documentation model string and factor levels",
    }
    # The frozen script uses the default separators of json.dumps(sort_keys=True), so byte-level compatibility must be preserved.
    raw = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

# Rerun Kahn's algorithm using the four protocol-declared priority rules to verify that the stored beta orders have not drifted through manual editing.
def regenerate_beta_from_kahn_rule(
    parents: Mapping[Node, Sequence[Node]],
    generation_rule: str,
) -> tuple[Node, ...]:
    """Rerun Kahn's algorithm using the frozen priority rule instead of blindly trusting the beta stored in JSON."""
    # The child sets are used both to update indegrees and as sorting keys for the minimum/maximum out-degree priority rules.
    children = {node: set() for node in parents}
    indegree = {node: len(tuple(node_parents)) for node, node_parents in parents.items()}
    for child, node_parents in parents.items():
        for parent in node_parents:
            if parent not in children:
                raise ValueError(f"Unknown parent {parent!r} of node {child!r}")
            children[parent].add(child)

    #  available always contains only currently zero-indegree nodes that have not yet been output.
    available = [node for node, degree in indegree.items() if degree == 0]
    regenerated: list[Node] = []
    while available:
        if generation_rule == "available_min_outdegree_then_name":
            selected = min(available, key=lambda node: (len(children[node]), node))
        elif generation_rule == "available_reverse_lexicographic":
            selected = max(available)
        elif generation_rule == "available_max_outdegree_then_name":
            selected = max(available, key=lambda node: (len(children[node]), node))
        elif generation_rule == "available_lexicographic":
            selected = min(available)
        else:
            raise ValueError(f"Unknown frozen Kahn generation rule: {generation_rule}")

        # Output the selected node and remove all of its outgoing edges; any newly created zero-indegree nodes enter the candidate set.
        available.remove(selected)
        regenerated.append(selected)
        for child in children[selected]:
            indegree[child] -= 1
            if indegree[child] == 0:
                available.append(child)

    if len(regenerated) != len(parents):
        raise ValueError("Frozen parent map is cyclic or incomplete")
    return tuple(regenerated)

# Return an independent copy of the quick, pilot, or formal settings so callers cannot mutate the global configuration.
def get_settings(mode: str) -> dict[str, object]:
    if mode not in EXPERIMENT_SETTINGS:
        raise ValueError(f"Unknown mode {mode!r}; choose quick, pilot, or formal")
    return dict(EXPERIMENT_SETTINGS[mode])

# Execute all hard assertions for the frozen header in one place; any failure must prevent Notebooks 01–03 from continuing.
def validate_frozen_header() -> None:
    """Validate the network and protocol header that must remain fixed before any probability computation."""
    
    if tuple(PARENT_ORDER) != NODE_ORDER or tuple(STATE_ORDER) != NODE_ORDER:
        raise AssertionError("State and parent maps must share the frozen node order")
    if len(NODE_ORDER) != 37 or len(ALARM_EDGES) != 46:
        raise AssertionError("ALARM must contain 37 nodes and 46 arcs")
    if int(NETWORK["original_parameter_dimension"]) != 509:
        raise AssertionError("Frozen ALARM parameter dimension must be 509")
    if frozen_network_semantic_sha256() != NETWORK["canonical_structure_state_sha256"]:
        raise AssertionError("ALARM structure/state semantic hash mismatch")
    if BASE_SEED != 65432 or not math.isclose(ISS, 10.0):
        raise AssertionError("Frozen seed or BDeu ISS changed")
    if CONDITION != {"PRSS": "NORMAL", "ECO2": "NORMAL"}:
        raise AssertionError("Frozen conditioning event changed")
    if TARGET != ("INT", "DISC", "MVS"):
        raise AssertionError("Frozen target changed")
    if TARGET_EVENT != { "INT": "ONESIDED", "DISC": "TRUE", "MVS": "NORMAL",}:
        raise AssertionError("Frozen target event changed")
    # The target event must assign every variable in TARGET, and no variables outside TARGET.
    if set(TARGET_EVENT) != set(TARGET):
        raise AssertionError( "TARGET_EVENT must assign exactly every TARGET variable" )
    # Every target state must belong to the frozen ALARM state space.
    for node, state in TARGET_EVENT.items():
        if node not in STATE_ORDER:
            raise AssertionError(f"Unknown target node: {node}")
        if state not in STATE_ORDER[node]:
            raise AssertionError(f"Invalid target state: {node}={state}")
    # The intervention experiment requires that TARGET contain none of the intervention nodes that are fixed and then removed.
    if set(TARGET) & set(INTERVENTION):
        raise AssertionError("TARGET must not contain an intervention-fixed node")
    if len(BETA_ORDERS) != 4 or len(set(BETA_ORDERS.values())) != 4:
        raise AssertionError("Exactly four distinct frozen topological orders are required")

    # Every stored beta must be reproducible from the current parent map using its declared Kahn generation rule.
    for record in ORDER_RECORDS:
        order_id = f"order_{record['id']}"
        #  Each time Notebook 00 is loaded, rerun the declared Kahn rule and compare the regenerated order with the frozen beta node by node.
        regenerated = regenerate_beta_from_kahn_rule(
            PARENT_ORDER,
            str(record["generation_rule"]),
        )
        if BETA_ORDERS[order_id] != regenerated:
            raise AssertionError(f"{order_id} no longer matches its frozen Kahn rule")

    # do-beta is not an independently hard-coded order; it must equal the restriction of the original beta after removing all intervention nodes.
    do_records_by_id = {
        f"order_{record['id']}": record
        for record in DO_ORDER_RECORDS
    }
    if set(do_records_by_id) != set(BETA_ORDERS):
        raise AssertionError("Observational and interventional order ids differ")
    intervention_nodes = set(INTERVENTION)
    for order_id, beta in BETA_ORDERS.items():
        # do-beta must be derived automatically by removing the intervention nodes from the observational beta and must not be hard-coded separately.
        expected_beta_do = tuple(
            node for node in beta
            if node not in intervention_nodes
        )
        if DO_BETA_ORDERS[order_id] != expected_beta_do:
            raise AssertionError(f"{order_id} do-beta is not beta restricted to V_do")
    formal = EXPERIMENT_SETTINGS["formal"]
    if formal != {
        "parameter_ids": tuple(range(100)),
        "sample_sizes": (100, 500, 1000, 2500, 5000, 7500, 10000),
        "repetitions": 100,
    }:
        raise AssertionError("Frozen formal experiment scale changed")
    if FROZEN_PROTOCOL["intervention"]["transform"] != (
        "generic_child_slicing_then_remove_intervention_nodes"
    ):
        raise AssertionError("Frozen intervention transform changed")

#  Execute hard assertions immediately when the module is loaded so that protocol drift is exposed before any experiment begins.
validate_frozen_header()


## 2. Common Excel, Hashing, and Runtime Environment Utilities

In [2]:
# # All output workbooks use the same readable style without changing any numerical content.
HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
HEADER_FONT = Font(color="FFFFFF", bold=True)
SUBHEADER_FILL = PatternFill("solid", fgColor="D9EAF7")

# Generate a timezone-aware UTC timestamp for auditing result workbooks.
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

# Record Python, dependency, platform, and CPU information that may affect reproducibility.
def environment_record() -> dict[str, object]:
    return {
        "timestamp_utc": utc_now_iso(),
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "platform": platform.platform(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
        "project_root": str(PROJECT_ROOT),
    }

# Normalize a DataFrame into CSV text and compute a stable content hash.
def dataframe_sha256(frame: pd.DataFrame, sort_by: Sequence[str] | None = None) -> str:
    stable = frame.copy()
    if sort_by:
        stable = stable.sort_values(list(sort_by), kind="mergesort")
    stable = stable.reset_index(drop=True)
    raw = stable.to_csv(index=False, lineterminator="\n", float_format="%.17g")
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

# Hash only code cells with the specified tag, so plotting or self-check changes do not affect the shared computation identity.
def tagged_code_sha256(notebook_path: Path, tag: str) -> str:
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    tagged_source = "\n".join(
        "".join(cell.get("source", []))
        for cell in notebook["cells"]
        if cell.get("cell_type") == "code"
        and tag in cell.get("metadata", {}).get("tags", [])
    )
    if not tagged_source.strip():
        raise ValueError(f"No code cells tagged {tag!r} in {notebook_path}")
    return hashlib.sha256(tagged_source.encode("utf-8")).hexdigest()

#  Return the unified code fingerprint of all common_core cells in Notebook 00.
def common_code_sha256() -> str:
    return (
        tagged_code_sha256(COMMON_NOTEBOOK, "common_core")
        if COMMON_NOTEBOOK.exists()
        else "not_available"
    )

# Aggregate the network, machine-readable protocol, and shared-code identity hashes.
def protocol_hashes() -> dict[str, str]:
    return {
        "network_semantic_sha256": NETWORK["canonical_structure_state_sha256"],
        "frozen_protocol_file_sha256": file_sha256(PROTOCOL_PATH),
        "common_core_file_sha256": common_code_sha256(),
    }

# Generate the protocol and runtime-environment metadata table written by every experiment.
def protocol_metadata(experiment: str, mode: str) -> pd.DataFrame:
    settings = get_settings(mode)
    record = {
        "schema_version": ALARM_SCHEMA_VERSION,
        "experiment": experiment,
        "mode": mode,
        "protocol_status": FROZEN_PROTOCOL["protocol_status"],
        "base_seed": BASE_SEED,
        "rng": FROZEN_PROTOCOL["rng"],
        "iss": ISS,
        "condition_json": json.dumps(CONDITION, ensure_ascii=False, sort_keys=True),
        "target_json": json.dumps(TARGET, ensure_ascii=False),
        "target_event_json": json.dumps(TARGET_EVENT, ensure_ascii=False, sort_keys=True),
        "intervention_json": json.dumps(INTERVENTION, ensure_ascii=False, sort_keys=True),
        "parameter_ids_json": json.dumps(settings["parameter_ids"]),
        "sample_sizes_json": json.dumps(settings["sample_sizes"]),
        "repetitions": settings["repetitions"],
        **protocol_hashes(),
        **environment_record(),
    }
    return pd.DataFrame([record])

# Compute relative bias only when the ground truth is sufficiently far from zero, avoiding numerically misleading ratios.
def safe_relative_bias(
    error: float,
    truth: float,
    minimum_denominator: float = 1e-12,
) -> float:
     """Return error/truth when the ground truth is sufficiently large; otherwise return NaN to avoid amplifying meaningless rounding errors."""
    return float(error / truth) if abs(truth) >= minimum_denominator else np.nan

#Combine settings, parameter-bank identity, structure hashes, and two levels of code hashes into an experiment-protocol fingerprint.
def experiment_protocol_sha256(
    experiment_id: int,
    mode: str,
    parameter_bank_sha256: str,
    structure_hashes: Mapping[str, str],
    experiment_payload: Mapping[str, object],
    experiment_code_sha256: str,
) -> str:
    """Encode all settings that affect the numerical meaning of the experiment into a stable protocol fingerprint."""
    payload = {
        "schema_version": ALARM_SCHEMA_VERSION,
        "experiment_id": int(experiment_id),
        "mode": mode,
        "settings": get_settings(mode),
        "base_seed": BASE_SEED,
        "iss": ISS,
        "network_semantic_sha256": NETWORK["canonical_structure_state_sha256"],
        "condition": CONDITION,
        "target": TARGET,
        "target_event": TARGET_EVENT,
        "intervention": INTERVENTION,
        "parameter_bank_sha256": parameter_bank_sha256,
        "structure_hashes": dict(structure_hashes),
        "experiment_payload": dict(experiment_payload),
        "common_code_sha256": common_code_sha256(),
        "experiment_code_sha256": experiment_code_sha256,
    }
    return hashlib.sha256(canonical_json_bytes(payload)).hexdigest()

#  Generate a one-row protocol worksheet containing the complete experiment identity.
def experiment_protocol_frame(
    experiment_id: int,
    mode: str,
    parameter_bank_sha256: str,
    structure_hashes: Mapping[str, str],
    experiment_payload: Mapping[str, object],
    experiment_code_sha256: str,
    protocol_sha256: str,
) -> pd.DataFrame:
     """Generate a one-row protocol table containing shared metadata, parameter-bank identity, structure identity, and experiment-code identity."""
    frame = protocol_metadata(f"experiment_{experiment_id}", mode)
    frame["parameter_bank_content_sha256"] = parameter_bank_sha256
    frame["structure_hashes_json"] = json.dumps(
        dict(structure_hashes), ensure_ascii=False, sort_keys=True
    )
    frame["experiment_payload_json"] = json.dumps(
        dict(experiment_payload), ensure_ascii=False, sort_keys=True
    )
    frame["experiment_code_sha256"] = experiment_code_sha256
    frame["protocol_sha256"] = protocol_sha256
    return frame

# Read back the protocol hash after writing to confirm that the result file belongs to the current run.
def verify_output_protocol(path: Path, expected_protocol_sha256: str) -> None:
     """Read back the protocol sheet from the result workbook and reject stale files that do not belong to the current run."""
    frame = pd.read_excel(path, sheet_name="protocol")
    if len(frame) != 1 or frame.loc[0, "protocol_sha256"] != expected_protocol_sha256:
        raise RuntimeError("Output workbook protocol hash mismatch")

#  Apply consistent Excel headers, freeze panes, filters, and reasonable column widths.
def format_workbook(path: Path) -> None:
    workbook = load_workbook(path)
    for sheet in workbook.worksheets:
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
        for cell in sheet[1]:
            cell.fill = HEADER_FILL
            cell.font = HEADER_FONT
            cell.alignment = Alignment(horizontal="center", vertical="center")
        for column in sheet.columns:
            letter = get_column_letter(column[0].column)
            width = max(
                10,
                min(60, max(len(str(cell.value or "")) for cell in column) + 2),
            )
            sheet.column_dimensions[letter].width = width
    workbook.save(path)

# Write to a temporary workbook first and then replace atomically, preventing partially written result files if execution is interrupted.
def write_excel_atomic(path: Path, sheets: Mapping[str, pd.DataFrame]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f"{path.stem}.tmp{path.suffix}")
    with pd.ExcelWriter(temporary, engine="openpyxl") as writer:
        for sheet_name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    format_workbook(temporary)
    temporary.replace(path)


## 3. DAG and FGCR

In [3]:
# Immutable DAG container; validate endpoints, duplicate nodes, and directed cycles immediately upon construction.
@dataclass(frozen=True)
class DAG:
      """Immutable DAG with a stable node order."""

    name: str
    nodes: tuple[Node, ...]
    edges: frozenset[Edge]

    # Validate object invariants immediately after construction, preventing invalid graphs from entering subsequent computations.
    def __post_init__(self) -> None:
        if len(set(self.nodes)) != len(self.nodes):
            raise ValueError("DAG nodes contain duplicates")
        node_set = set(self.nodes)
        for edge in self.edges:
            if len(edge) != 2:
                raise ValueError(f"Illegal edge: {edge!r}")
            parent, child = edge
            if parent == child or parent not in node_set or child not in node_set:
                raise ValueError(f"Illegal edge: {edge!r}")
        if not is_dag(self):
            raise ValueError(f"{self.name} contains a directed cycle")

    #  Return parent nodes in the stable node order to ensure reproducible structural output and hashes.
    def parents(self, node: Node) -> tuple[Node, ...]:
        if node not in self.nodes:
            raise KeyError(node)
        position = {value: index for index, value in enumerate(self.nodes)}
        return tuple(sorted((u for u, v in self.edges if v == node), key=position.get))

    # Return child nodes in the stable node order so that generic interventions can automatically identify affected CPTs.
    def children(self, node: Node) -> tuple[Node, ...]:
        if node not in self.nodes:
            raise KeyError(node)
        position = {value: index for index, value in enumerate(self.nodes)}
        return tuple(sorted((v for u, v in self.edges if u == node), key=position.get))

# Use Kahn's algorithm to determine whether all nodes can be removed topologically.
def is_dag(graph: DAG) -> bool:
    indegree = {node: 0 for node in graph.nodes}
    children = {node: [] for node in graph.nodes}
    for parent, child in graph.edges:
        indegree[child] += 1
        children[parent].append(child)
    queue = [node for node in graph.nodes if indegree[node] == 0]
    visited = 0
    while queue:
        node = queue.pop(0)
        visited += 1
        for child in children[node]:
            indegree[child] -= 1
            if indegree[child] == 0:
                queue.append(child)
    return visited == len(graph.nodes)

#  Generate a reproducible topological order using graph.nodes as the stable priority order.
def topological_sort(graph: DAG) -> tuple[Node, ...]:
    position = {node: index for index, node in enumerate(graph.nodes)}
    indegree = {node: 0 for node in graph.nodes}
    children = {node: [] for node in graph.nodes}
    for parent, child in graph.edges:
        indegree[child] += 1
        children[parent].append(child)
    ready = sorted((node for node in graph.nodes if indegree[node] == 0), key=position.get)
    result: list[Node] = []
    while ready:
        node = ready.pop(0)
        result.append(node)
        for child in sorted(children[node], key=position.get):
            indegree[child] -= 1
            if indegree[child] == 0:
                ready.append(child)
                ready.sort(key=position.get)
    if len(result) != len(graph.nodes):
        raise ValueError(f"{graph.name} is not acyclic")
    return tuple(result)

# Verify that a candidate order contains all nodes and directs every edge from earlier to later.
def validate_topological_order(graph: DAG, order: Sequence[Node]) -> None:
    order = tuple(order)
    if len(order) != len(graph.nodes) or set(order) != set(graph.nodes):
        raise ValueError("Order must be a permutation of all graph nodes")
    position = {node: index for index, node in enumerate(order)}
    violations = [(u, v) for u, v in graph.edges if position[u] >= position[v]]
    if violations:
        raise ValueError(f"Order is not topological; first violations: {violations[:5]}")

# Take the induced subgraph on the specified node set while preserving the original relative node order.
def induced_subgraph(graph: DAG, keep: Iterable[Node], name: str | None = None) -> DAG:
    keep_set = set(keep)
    nodes = tuple(node for node in graph.nodes if node in keep_set)
    edges = frozenset(
        (u, v) for u, v in graph.edges if u in keep_set and v in keep_set
    )
    return DAG(name or f"{graph.name}_sub", nodes, edges)

#  Convert graph parent sets into a deterministic parent-axis mapping ordered according to graph.nodes.
def parent_order_from_graph(graph: DAG) -> ParentOrder:
    return {node: graph.parents(node) for node in graph.nodes}

# Canonically convert a parent mapping into a directed edge set.
def edge_set_from_parents(parents: Mapping[Node, Sequence[Node]]) -> frozenset[Edge]:
    return frozenset((parent, node) for node, values in parents.items() for parent in values)

# Instantiate the unique original graph from the frozen parent sets; the original CPT parent axes remain defined by PARENT_ORDER.
ALARM_DAG = DAG("ALARM", NODE_ORDER, ALARM_EDGES)
if edge_set_from_parents(PARENT_ORDER) != ALARM_DAG.edges:
    raise AssertionError("Frozen PARENT_ORDER does not match the ALARM DAG")

# Store alpha_C, alpha_R, and alpha obtained by partitioning beta in Algorithm 1.
@dataclass(frozen=True)
class Algorithm1Orders:
    beta: tuple[Node, ...]
    alpha_C: tuple[Node, ...]
    alpha_R: tuple[Node, ...]
    alpha: tuple[Node, ...]

# Package the FGCR output graphs, fill-in edges, and execution counts for covered reversals.
@dataclass(frozen=True)
class FGCRResult:
    orders: Algorithm1Orders
    H: DAG
    H_R: DAG
    added_edges_R: tuple[Edge, ...]
    n_swaps: int
    n_reversals: int
    n_cover_additions: int

#  Preserve the relative order within beta while moving conditioning nodes to the front of alpha.
def construct_algorithm1_orders(
    graph: DAG,
    conditioning_nodes: Iterable[Node],
    beta: Sequence[Node],
) -> Algorithm1Orders:
    validate_topological_order(graph, beta)
    condition_set = set(conditioning_nodes)
    if not condition_set.issubset(set(graph.nodes)):
        raise ValueError("Conditioning set contains nodes outside the graph")
    alpha_c = tuple(node for node in beta if node in condition_set)
    alpha_r = tuple(node for node in beta if node not in condition_set)
    return Algorithm1Orders(tuple(beta), alpha_c, alpha_r, alpha_c + alpha_r)

# Quickly retrieve the parent set of a node from the mutable edge set used by FGCR.
def _parents_from_edges(edges: set[Edge], node: Node) -> set[Node]:
    return {parent for parent, child in edges if child == node}

# Add the required edges so that source->target becomes covered, then reverse it safely.
def _cover_and_reverse(edges: set[Edge], source: Node, target: Node) -> int:
    if (source, target) not in edges:
        raise ValueError(f"Missing edge {source}->{target}")
    pa_source = _parents_from_edges(edges, source)
    pa_target = _parents_from_edges(edges, target)
    additions = {(node, target) for node in pa_source - pa_target}
    additions |= {(node, source) for node in pa_target - pa_source - {source}}
    before = len(edges)
    edges.update(additions)
    if _parents_from_edges(edges, target) != _parents_from_edges(edges, source) | {source}:
        raise AssertionError("Edge was not covered before reversal")
    edges.remove((source, target))
    edges.add((target, source))
    return len(edges) - before

# Execute Algorithm 1 and Algorithm 2 to obtain the full graph H and the retained conditional graph H_R.
def fgcr(
    graph: DAG,
    conditioning_nodes: Iterable[Node],
    beta: Sequence[Node],
) -> FGCRResult:
     """Algorithm 1 + Method B2; return the full H and H_R on the nodes retained after conditioning."""
    # First construct the target order alpha=(alpha_C, alpha_R), then begin the edge-reversal process.
    orders = construct_algorithm1_orders(graph, conditioning_nodes, beta)
    beta_current = list(orders.beta)
    alpha_position = {node: index for index, node in enumerate(orders.alpha)}
    beta_position = {node: index for index, node in enumerate(beta_current)}
    edges = set(graph.edges)
    n_swaps = n_reversals = n_cover_additions = 0

   # Algorithm 2 positions nodes from right to left in alpha, ensuring that the already positioned suffix never changes again.
    for node in reversed(orders.alpha):
        while beta_position[node] < len(beta_current) - 1:
            index = beta_position[node]
            right = beta_current[index + 1]
            if alpha_position[right] > alpha_position[node]:
                break
             # Perform covering and covered-arc reversal only when the adjacent swap would reverse an existing edge.
            if (node, right) in edges:
                n_cover_additions += _cover_and_reverse(edges, node, right)
                n_reversals += 1
            beta_current[index], beta_current[index + 1] = right, node
            beta_position[right] = index
            beta_position[node] = index + 1
            n_swaps += 1

    if tuple(beta_current) != orders.alpha:
        raise AssertionError("FGCR failed to reach alpha")
   # The edge set after all swaps defines the full minimal I-map H; then take its induced subgraph on R to obtain H_R.
    H = DAG(f"{graph.name}_H", orders.alpha, frozenset(edges))
    H_R = induced_subgraph(H, orders.alpha_R, f"{graph.name}_H_R")
    G_R = induced_subgraph(graph, orders.alpha_R, f"{graph.name}_G_R")
    if not G_R.edges.issubset(H_R.edges):
        raise AssertionError("FGCR output H_R does not contain G_R")
    added_edges = tuple(
        sorted(H_R.edges - G_R.edges, key=lambda edge: (orders.alpha_R.index(edge[0]), orders.alpha_R.index(edge[1])))
    )
    return FGCRResult(
        orders=orders,
        H=H,
        H_R=H_R,
        added_edges_R=added_edges,
        n_swaps=n_swaps,
        n_reversals=n_reversals,
        n_cover_additions=n_cover_additions,
    )

#  Compute the number of free parameters using the multi-state BN formula Σ(r_v−1)q_v.
def parameter_dimension(
    graph_or_parents: DAG | Mapping[Node, Sequence[Node]],
    states: Mapping[Node, Sequence[str]] = STATE_ORDER,
) -> int:
    if isinstance(graph_or_parents, DAG):
        nodes = graph_or_parents.nodes
        parents = parent_order_from_graph(graph_or_parents)
    else:
        parents = graph_or_parents
        nodes = tuple(parents)
    total = 0
    for node in nodes:
        parent_configs = math.prod(len(states[parent]) for parent in parents[node])
        total += (len(states[node]) - 1) * parent_configs
    return total


## 4. Generic Multi-State Exact Probability Engine
This section provides factor operations and variable elimination for arbitrary finite cardinalities. It also retains an explicit ancestral-subgraph enumeration path that does not reuse the factor multiplication or elimination code, serving as an independent oracle. The two paths must agree in the self-checks performed in Notebook 00.


In [4]:
# Represent an ordered variable scope and its multidimensional probability array; this is the basic computational object used by VE.
@dataclass(frozen=True)
class Factor:
    variables: tuple[Node, ...]
    values: np.ndarray

    # Validate object invariants immediately after construction, preventing invalid graphs or factors from entering subsequent computations.
    def __post_init__(self) -> None:
        values = np.asarray(self.values, dtype=float)
        if values.ndim != len(self.variables):
            raise ValueError("Factor dimension does not match its variable tuple")
        if len(set(self.variables)) != len(self.variables):
            raise ValueError("Factor variables contain duplicates")
        object.__setattr__(self, "values", values)

    # Fix the corresponding axes according to the evidence indices and remove those variables from the factor scope.
    def reduce(self, evidence: Mapping[Node, int]) -> "Factor":
        variables = list(self.variables)
        values = self.values
        for variable in tuple(self.variables):
            if variable in evidence:
                axis = variables.index(variable)
                values = np.take(values, int(evidence[variable]), axis=axis)
                variables.pop(axis)
        return Factor(tuple(variables), values)

    # Sum along the specified variable axis to perform the marginalization step of variable elimination.
    def sum_out(self, variable: Node) -> "Factor":
        axis = self.variables.index(variable)
        variables = self.variables[:axis] + self.variables[axis + 1:]
        return Factor(variables, self.values.sum(axis=axis))

# Add singleton axes and reorder dimensions so that factors with different scopes can be multiplied by broadcasting.
def _align_factor(factor: Factor, all_variables: Sequence[Node]) -> np.ndarray:
    present = [node for node in all_variables if node in factor.variables]
    permutation = [factor.variables.index(node) for node in present]
    values = np.transpose(factor.values, permutation) if permutation else factor.values
    shape = [CARDINALITY[node] if node in factor.variables else 1 for node in all_variables]
    return values.reshape(shape)

# Take the union of the scopes of two factors and perform aligned elementwise multiplication.
def multiply_two(first: Factor, second: Factor) -> Factor:
    all_variables = tuple(dict.fromkeys(first.variables + second.variables))
    values = _align_factor(first, all_variables) * _align_factor(second, all_variables)
    return Factor(all_variables, values)

#Preferentially merge smaller factors to reduce the peak size of intermediate tensors.
def multiply_all(factors: Sequence[Factor]) -> Factor:
    if not factors:
        return Factor((), np.array(1.0))
    work = list(factors)
    while len(work) > 1:
        work.sort(key=lambda factor: factor.values.size)
        first, second = work.pop(0), work.pop(0)
        work.append(multiply_two(first, second))
    return work[0]

# Determine a reproducible elimination order using min-fill, state-space size, and the frozen node order.
def elimination_order(factors: Sequence[Factor], candidates: Iterable[Node]) -> list[Node]:
    scopes = [set(factor.variables) for factor in factors]
    remaining = set(candidates)
    result: list[Node] = []
    node_position = {node: index for index, node in enumerate(NODE_ORDER)}
    while remaining:
        scores: list[tuple[int, int, int, Node]] = []
        existing = {
            frozenset((left, right))
            for scope in scopes
            for left, right in combinations(sorted(scope), 2)
        }
        for variable in remaining:
            involved = [scope for scope in scopes if variable in scope]
            neighbors = set().union(*involved) - {variable} if involved else set()
            fill = sum(
                frozenset((left, right)) not in existing
                for left, right in combinations(sorted(neighbors), 2)
            )
            state_space = math.prod(CARDINALITY[node] for node in neighbors)
            scores.append((fill, state_space, node_position.get(variable, 10**9), variable))
        variable = min(scores)[-1]
        involved = [scope for scope in scopes if variable in scope]
        scopes = [scope for scope in scopes if variable not in scope]
        if involved:
            scopes.append(set().union(*involved) - {variable})
        remaining.remove(variable)
        result.append(variable)
    return result

# Convert state-name assignments into CPT array indices, rejecting unknown nodes or states.
def _validate_name_assignment(assignments: Mapping[Node, str]) -> dict[Node, int]:
    indices: dict[Node, int] = {}
    for node, state in assignments.items():
        if node not in STATE_ORDER:
            raise KeyError(f"Unknown node {node!r}")
        if state not in STATE_TO_INDEX[node]:
            raise ValueError(f"Unknown state {state!r} for {node}")
        indices[node] = STATE_TO_INDEX[node][state]
    return indices

# Verify node sets, parent axes, shapes, non-negativity, and normalization along the state axis.
def validate_cpts(
    graph: DAG,
    cpts: Mapping[Node, np.ndarray],
    parent_order: Mapping[Node, Sequence[Node]],
    atol: float = 1e-12,
) -> None:
    if set(cpts) != set(graph.nodes) or set(parent_order) != set(graph.nodes):
        raise ValueError("CPT and parent-order node sets must equal graph.nodes")
    for node in graph.nodes:
        parents = tuple(parent_order[node])
        if len(set(parents)) != len(parents) or set(parents) != set(graph.parents(node)):
            raise ValueError(f"Parent-axis definition does not match graph for {node}")
        expected_shape = (CARDINALITY[node], *(CARDINALITY[p] for p in parents))
        values = np.asarray(cpts[node], dtype=float)
        if values.shape != expected_shape:
            raise ValueError(f"CPT shape mismatch for {node}: {values.shape} != {expected_shape}")
        if not np.all(np.isfinite(values)) or np.any(values < 0):
            raise ValueError(f"CPT for {node} contains invalid probabilities")
        if not np.allclose(values.sum(axis=0), 1.0, atol=atol, rtol=0.0):
            raise ValueError(f"CPT for {node} is not normalized along its state axis")

# Convert each local CPT into a factor with variable order: node, parents...
def cpt_factors(
    cpts: Mapping[Node, np.ndarray],
    parent_order: Mapping[Node, Sequence[Node]],
) -> list[Factor]:
    return [Factor((node, *parent_order[node]), cpts[node]) for node in parent_order]

# Perform generic multi-state variable elimination and return the normalized distribution in the requested variable order.
def ve_query(
    cpts: Mapping[Node, np.ndarray],
    parent_order: Mapping[Node, Sequence[Node]],
    query_variables: Sequence[Node],
    evidence_names: Mapping[Node, str] | None = None,
) -> np.ndarray:
    query = tuple(query_variables)
    if not query or len(set(query)) != len(query):
        raise ValueError("Query variables must be non-empty and distinct")
    evidence_names = dict(evidence_names or {})
    if set(query) & set(evidence_names):
        raise ValueError("A query variable cannot simultaneously be evidence")
    evidence = _validate_name_assignment(evidence_names)
    # Absorb evidence into each local factor first so that subsequent tensors are reduced immediately.
    factors = [factor.reduce(evidence) for factor in cpt_factors(cpts, parent_order)]
    candidates = set(parent_order) - set(query) - set(evidence)
    # Following the frozen min-fill order, collect, multiply, and sum out variables one by one.
    for variable in elimination_order(factors, candidates):
        involved = [factor for factor in factors if variable in factor.variables]
        factors = [factor for factor in factors if variable not in factor.variables]
        if involved:
            factors.append(multiply_all(involved).sum_out(variable))
    result = multiply_all(factors)
    for variable in tuple(result.variables):
        if variable not in query:
            result = result.sum_out(variable)
    if set(result.variables) != set(query):
        raise AssertionError("VE result lost a query variable")
    permutation = [result.variables.index(node) for node in query]
    values = np.transpose(result.values, permutation)
    normalizer = float(values.sum())
    if not np.isfinite(normalizer) or normalizer <= 0:
        raise FloatingPointError("Invalid VE normalization constant")
    return values / normalizer

# Independently enumerate the ancestral subgraph of query and evidence as an oracle that shares no VE code.
def ancestral_enumeration_query(
    cpts: Mapping[Node, np.ndarray],
    graph: DAG,
    parent_order: Mapping[Node, Sequence[Node]],
    query_variables: Sequence[Node],
    evidence_names: Mapping[Node, str] | None = None,
) -> np.ndarray:
    """Independent oracle: explicitly enumerate the ancestral subgraph of query∪evidence without calling Factor/VE."""
    query = tuple(query_variables)
    evidence_names = dict(evidence_names or {})
    _validate_name_assignment(evidence_names)
     # The independent oracle enumerates only the ancestors of query∪evidence, avoiding explosion over the full 37-node state space.
    relevant = set(query) | set(evidence_names)
    stack = list(relevant)
    while stack:
        node = stack.pop()
        for parent in parent_order[node]:
            if parent not in relevant:
                relevant.add(parent)
                stack.append(parent)
    order = tuple(node for node in topological_sort(graph) if node in relevant)
    free = tuple(node for node in order if node not in evidence_names)
    free_cards = tuple(CARDINALITY[node] for node in free)
    # Construct the Cartesian assignment grid for all free ancestral variables at once.
    grids = np.indices(free_cards, dtype=np.int16).reshape((len(free), -1))
    assignment: dict[Node, np.ndarray] = {
        node: grids[index] for index, node in enumerate(free)
    }
    n_assignments = grids.shape[1]
    for node, state in evidence_names.items():
        assignment[node] = np.full(
            n_assignments,
            STATE_TO_INDEX[node][state],
            dtype=np.int16,
        )
    #  Multiply all local CPT contributions in topological order to obtain the joint weight of each complete assignment.
    weights = np.ones(n_assignments, dtype=float)
    for node in order:
        index = (assignment[node], *(assignment[parent] for parent in parent_order[node]))
        weights *= cpts[node][index]
    flat_query_index = np.zeros(n_assignments, dtype=np.int64)
    query_cards: list[int] = []
    for node in query:
        query_cards.append(CARDINALITY[node])
        flat_query_index = flat_query_index * CARDINALITY[node] + assignment[node]
    values = np.bincount(
        flat_query_index,
        weights=weights,
        minlength=math.prod(query_cards),
    ).reshape(tuple(query_cards))
    normalizer = float(values.sum())
    if not np.isfinite(normalizer) or normalizer <= 0:
        raise FloatingPointError("Invalid enumeration normalization constant")
    return values / normalizer

# Compile P(V) into local CPTs for P(R|C) on the target H_R using exact joint family distributions.
def compile_conditional_cpts(
    source_cpts: Mapping[Node, np.ndarray],
    source_parent_order: Mapping[Node, Sequence[Node]],
    target_graph: DAG,
    condition: Mapping[Node, str],
) -> tuple[CPTMap, ParentOrder]:
    """Exactly compile P(V) into local CPTs of P(R|condition) on the target graph."""
    target_parent_order = parent_order_from_graph(target_graph)
    compiled: CPTMap = {}
    for node in target_graph.nodes:
        ## Obtain each target local CPT from the exact conditional joint distribution of the node and its H_R parent set.
        family = (node, *target_parent_order[node])
        joint = ve_query(source_cpts, source_parent_order, family, condition)
        denominator = joint.sum(axis=0, keepdims=True)
        if np.any(denominator <= 0):
            raise FloatingPointError(f"Zero-probability parent configuration for {node}")
        compiled[node] = joint / denominator
    validate_cpts(target_graph, compiled, target_parent_order)
    return compiled, target_parent_order

# Compute the TVD and maximum cell-wise absolute error between two discrete distributions with the same shape.
def distribution_errors(reference: np.ndarray, estimate: np.ndarray) -> dict[str, float]:
    reference = np.asarray(reference, dtype=float)
    estimate = np.asarray(estimate, dtype=float)
    if reference.shape != estimate.shape:
        raise ValueError("Distribution shapes differ")
    delta = np.abs(reference - estimate)
    return {
        "tvd": float(0.5 * delta.sum()),
        "max_abs_error": float(delta.max(initial=0.0)),
    }

# Read the probability of a specified target event from the joint distribution using the frozen state order.
def event_probability(
    distribution: np.ndarray,
    query_variables: Sequence[Node],
    event: Mapping[Node, str],
) -> float:
    query = tuple(query_variables)
    if set(event) != set(query):
        raise ValueError("Event must assign exactly every query variable")
    index = tuple(STATE_TO_INDEX[node][event[node]] for node in query)
    return float(distribution[index])

# Include node names, parent-axis order, and IEEE-754 CPT bytes together in the parameter identity hash.
def cpts_sha256(
    cpts: Mapping[Node, np.ndarray],
    parent_order: Mapping[Node, Sequence[Node]],
) -> str:
    digest = hashlib.sha256()
    for node in parent_order:
        digest.update(node.encode("utf-8"))
        digest.update(b"|")
        digest.update(",".join(parent_order[node]).encode("utf-8"))
        digest.update(np.asarray(cpts[node], dtype="<f8").tobytes(order="C"))
    return digest.hexdigest()


## 5. Shared Dirichlet Parameter Bank for the Three Experiments
For each node and each parent configuration, parameters are independently sampled from $\mathrm{Dirichlet}(1,\ldots,1)$. Parameter set `parameter_id` is generated using  `PCG64(SeedSequence([65432, parameter_id]))`; the formal parameter IDs are 0–99. The long-format Excel table stores state names, parent configurations, and content hashes, allowing Notebooks 01–03 to verify that they are reading the same parameter sets.

In [5]:
# Store shared parameter IDs, CPT contents, and the logical content hash.
@dataclass(frozen=True)
class ParameterBank:
    parameter_ids: tuple[int, ...]
    values: Mapping[int, Mapping[Node, np.ndarray]]
    content_sha256: str

    #  Retrieve one CPT set by parameter ID; copy arrays by default to prevent callers from modifying the shared parameters.
    def cpts(self, parameter_id: int, copy: bool = True) -> CPTMap:
        if parameter_id not in self.values:
            raise KeyError(f"Parameter id {parameter_id} is absent from the bank")
        if copy:
            return {node: np.array(values, copy=True) for node, values in self.values[parameter_id].items()}
        return dict(self.values[parameter_id])

# Create the frozen PCG64 stream strictly from SeedSequence([65432, parameter_id]).
def parameter_rng(parameter_id: int) -> np.random.Generator:
    if not isinstance(parameter_id, (int, np.integer)) or int(parameter_id) < 0:
        raise ValueError("parameter_id must be a non-negative integer")
    # Each parameter_id uses an independent substream, so adding or removing other IDs does not change the parameters of the current set.
    seed_sequence = np.random.SeedSequence([BASE_SEED, int(parameter_id)]
    seed_sequence = np.random.SeedSequence([BASE_SEED, int(parameter_id)])
    return np.random.Generator(np.random.PCG64(seed_sequence))

#  Derive mutually independent and reproducible uint32 seeds for sample size, repetition, and data stream.
def derive_seed(
    experiment_id: int,
    parameter_id: int,
    sample_size: int,
    replicate: int,
    stream_id: int,
) -> int:
     """Stably derive uint32 seeds for the independent data streams in Notebooks 01–03."""
    values = [BASE_SEED, experiment_id, parameter_id, sample_size, replicate, stream_id]
    return int(np.random.SeedSequence(values).generate_state(1, dtype=np.uint32)[0])

# Independently draw a Dirichlet(1,...,1) row for every parent configuration of every node.
def generate_random_cpts(parameter_id: int) -> CPTMap:
    rng = parameter_rng(parameter_id)
    cpts: CPTMap = {}
    for node in NODE_ORDER:
        response_card = CARDINALITY[node]
        parent_cards = [CARDINALITY[parent] for parent in PARENT_ORDER[node]]
        n_parent_configs = math.prod(parent_cards)
        # Draw one strictly positive and normalized multi-state probability row for each parent configuration.
        rows = rng.dirichlet(np.ones(response_card), size=n_parent_configs)
        cpts[node] = rows.reshape((*parent_cards, response_card)).transpose(
            (-1, *range(len(parent_cards)))
        )
    validate_cpts(ALARM_DAG, cpts, PARENT_ORDER)
    return cpts

# Aggregate the CPT hashes in parameter-ID order to form the logical identity of the parameter bank.
def parameter_bank_content_sha256(
    values: Mapping[int, Mapping[Node, np.ndarray]],
) -> str:
    digest = hashlib.sha256()
    for parameter_id in sorted(values):
        digest.update(str(int(parameter_id)).encode("ascii"))
        digest.update(cpts_sha256(values[parameter_id], PARENT_ORDER).encode("ascii"))
    return digest.hexdigest()

# Generate the shared parameter sets for the specified IDs and immediately compute their content hash.
def generate_parameter_bank(parameter_ids: Iterable[int]) -> ParameterBank:
    ids = tuple(int(value) for value in parameter_ids)
    if len(set(ids)) != len(ids):
        raise ValueError("Parameter ids contain duplicates")
    values = {parameter_id: generate_random_cpts(parameter_id) for parameter_id in ids}
    return ParameterBank(ids, values, parameter_bank_content_sha256(values))

# Expand multidimensional CPTs into a long-format table and simultaneously generate the frozen metadata sheet.
def parameter_bank_frames(bank: ParameterBank) -> dict[str, pd.DataFrame]:
    rows: list[dict[str, object]] = []
    for parameter_id in bank.parameter_ids:
        cpts = bank.values[parameter_id]
        for node in NODE_ORDER:
            parents = PARENT_ORDER[node]
            parent_ranges = [range(CARDINALITY[parent]) for parent in parents]
            parent_configs = product(*parent_ranges) if parents else [()]
            for config_index, config in enumerate(parent_configs):
                config_names = {
                    parent: INDEX_TO_STATE[parent][state_index]
                    for parent, state_index in zip(parents, config)
                }
                for state_index, state in enumerate(STATE_ORDER[node]):
                    rows.append({
                        "parameter_id": parameter_id,
                        "node": node,
                        "parents_json": json.dumps(parents),
                        "parent_config_index": config_index,
                        "parent_config_json": json.dumps(config_names, sort_keys=True),
                        "state_index": state_index,
                        "state": state,
                        "probability": float(cpts[node][(state_index, *config)]),
                        # Hexadecimal floating-point text preserves the complete IEEE-754 identity that Excel numeric cells cannot guarantee.
                        "probability_hex": float(cpts[node][(state_index, *config)]).hex(),
                    })
    metadata = pd.DataFrame([{
        "schema_version": ALARM_SCHEMA_VERSION,
        "network": "ALARM",
        "protocol_status": FROZEN_PROTOCOL["protocol_status"],
        "network_semantic_sha256": NETWORK["canonical_structure_state_sha256"],
        "generator": "independent Dirichlet(1,...,1) per node/parent configuration",
        "rng": FROZEN_PROTOCOL["rng"],
        "base_seed": BASE_SEED,
        "parameter_ids_json": json.dumps(bank.parameter_ids),
        "content_sha256": bank.content_sha256,
        "created_utc": utc_now_iso(),
    }])
    return {"metadata": metadata, "cpt_values": pd.DataFrame(rows)}

# Save the parameter bank using the shared atomic-write utility.
def write_parameter_bank(path: Path, bank: ParameterBank) -> None:
    write_excel_atomic(path, parameter_bank_frames(bank))

# Reconstruct CPTs losslessly from Excel and verify the schema, parent configurations, normalization, and content hash.
def load_parameter_bank(path: Path) -> ParameterBank:
    metadata = pd.read_excel(path, sheet_name="metadata")
    values_frame = pd.read_excel(path, sheet_name="cpt_values")
    if len(metadata) != 1:
        raise ValueError("Parameter bank metadata must contain exactly one row")
    row = metadata.iloc[0]
    if int(row["schema_version"]) != ALARM_SCHEMA_VERSION:
        raise ValueError("Parameter bank schema version mismatch")
    if row["network_semantic_sha256"] != NETWORK["canonical_structure_state_sha256"]:
        raise ValueError("Parameter bank belongs to another ALARM definition")
    parameter_ids = tuple(int(value) for value in json.loads(row["parameter_ids_json"]))
    values: dict[int, CPTMap] = {}
    for parameter_id in parameter_ids:
        group = values_frame.loc[values_frame["parameter_id"] == parameter_id]
        cpts: CPTMap = {}
        for node in NODE_ORDER:
            node_rows = group.loc[group["node"] == node]
            expected_shape = (
                CARDINALITY[node],
                *(CARDINALITY[parent] for parent in PARENT_ORDER[node]),
            )
            array = np.empty(expected_shape, dtype=float)
            seen: set[tuple[int, ...]] = set()
            for record in node_rows.itertuples(index=False):
                config_names = json.loads(record.parent_config_json)
                config = tuple(
                    STATE_TO_INDEX[parent][config_names[parent]]
                    for parent in PARENT_ORDER[node]
                )
                index = (int(record.state_index), *config)
                if index in seen:
                    raise ValueError(f"Duplicate parameter-bank row for {parameter_id}/{node}/{index}")
                # Excel numeric cells preserve only about 15 decimal digits; the hex field stores the lossless IEEE-754 text identity.
                # Reconstructing from it keeps the shared parameter-bank hash unchanged across write/read round trips.
                probability = float.fromhex(str(record.probability_hex))
                if not math.isclose(
                    probability,
                    float(record.probability),
                    rel_tol=1e-14,
                    abs_tol=1e-15,
                ):
                    raise ValueError(
                        f"Decimal/hex probability mismatch for {parameter_id}/{node}/{index}"
                    )
                array[index] = probability
                seen.add(index)
            if len(seen) != array.size:
                raise ValueError(f"Incomplete parameter-bank rows for {parameter_id}/{node}")
            cpts[node] = array
        validate_cpts(ALARM_DAG, cpts, PARENT_ORDER)
        values[parameter_id] = cpts
    content_hash = parameter_bank_content_sha256(values)
    if content_hash != row["content_sha256"]:
        raise ValueError("Parameter bank content hash mismatch")
    return ParameterBank(parameter_ids, values, content_hash)

# Reuse the matching shared parameter bank; reject silent overwriting when the parameter IDs differ.
# Only the already frozen parameter bank may be loaded; under the formal protocol, missing files must never trigger random regeneration.
def ensure_shared_parameter_bank(
    path: Path = PARAMETER_BANK_PATH,
    parameter_ids: Iterable[int] = range(N_PARAMETER_SETS),
) -> ParameterBank:
    requested = tuple(int(value) for value in parameter_ids)

    # The parameter bank is a frozen input; stop immediately if it is missing to avoid regenerating byte-level-different results under another NumPy environment.
    if not path.exists():
        raise FileNotFoundError(
            "Frozen parameter bank is missing: "
            f"{path}. Restore data/alarm_parameter_sets.xlsx; "
            "do not regenerate it."
        )

    # load_parameter_bank has already verified the internal Excel content hash, CPT shapes, and probability normalization.
    bank = load_parameter_bank(path)

    # Additionally compare against the fixed hash in the machine-readable protocol so that another self-consistent but different parameter bank cannot be accepted.
    if bank.content_sha256 != FROZEN_PARAMETER_BANK_SHA256:
        raise ValueError(
            "Parameter bank does not match the frozen protocol: "
            f"expected {FROZEN_PARAMETER_BANK_SHA256}, "
            f"observed {bank.content_sha256}"
        )

     # The formal parameter bank must contain exactly IDs 0–99; parameter sets may not be silently omitted or added.
    if bank.parameter_ids != requested:
        raise ValueError(
            "Frozen parameter bank has different parameter ids"
        )

    return bank


## 6. Generic Deterministic-do Child Slicing

The shared function derives affected nodes only from `graph.children(a)` and the parent-axis definitions, supports one or multiple intervention variables, and contains no special-case handling for `VTUB`. The frozen protocol separately asserts that, in the current ALARM network,$\operatorname{ch}_G(VMCH)=\{VTUB\}$,so any unintended modification of the BIF file or graph structure causes an immediate failure.


In [6]:
# Package the graph, sliced CPTs, parent-axis definitions, and actually affected child nodes after removing the intervention nodes.
@dataclass(frozen=True)
class InterventionResult:
    graph: DAG
    cpts: Mapping[Node, np.ndarray]
    parent_order: Mapping[Node, tuple[Node, ...]]
    sliced_children: Mapping[Node, tuple[Node, ...]]
    assignments: Mapping[Node, str]

# Read the children of each intervention node directly from the current graph, without using any ALARM-specific special cases.
def intervention_children(
    graph: DAG,
    intervention_nodes: Iterable[Node],
) -> dict[Node, tuple[Node, ...]]:
    result: dict[Node, tuple[Node, ...]] = {}
    for node in intervention_nodes:
        if node not in graph.nodes:
            raise KeyError(f"Unknown intervention node {node!r}")
        result[node] = graph.children(node)
    return result

#  Protocol-level assertion that the intervention-related child set in the current ALARM graph still matches the frozen record.
def assert_frozen_intervention_children(graph: DAG = ALARM_DAG) -> None:
    actual = intervention_children(graph, INTERVENTION)
    if actual != EXPECTED_INTERVENTION_CHILDREN:
        raise AssertionError(
            "Frozen ALARM intervention structure changed: "
            f"expected {EXPECTED_INTERVENTION_CHILDREN}, got {actual}"
        )

# Fix all intervention parent axes, update the CPTs of all affected children, and then remove the intervention nodes.
def intervene_and_remove(
    graph: DAG,
    cpts: Mapping[Node, np.ndarray],
    parent_order: Mapping[Node, Sequence[Node]],
    assignments: Mapping[Node, str],
) -> InterventionResult:
    """Generic child slicing: fix all intervention parent axes, then remove the intervention nodes."""
    assignments = dict(assignments)
    if not assignments:
        raise ValueError("At least one deterministic intervention is required")
    _validate_name_assignment(assignments)
    if not set(assignments).issubset(set(graph.nodes)):
        raise ValueError("Intervention contains a node outside the current graph")
    validate_cpts(graph, cpts, parent_order)

    intervention_nodes = set(assignments)
    # Automatically identify all affected child nodes from the graph; the shared algorithm has no special knowledge of VTUB.
    sliced_children = intervention_children(graph, assignments)
    affected = set().union(*map(set, sliced_children.values())) - intervention_nodes
    remaining_nodes = tuple(node for node in graph.nodes if node not in intervention_nodes)
    # Structurally remove all edges incident to intervention nodes, retaining the subgraph over the stochastic nodes.
    remaining_edges = frozenset(
        (parent, child)
        for parent, child in graph.edges
        if parent not in intervention_nodes and child not in intervention_nodes
    )
    reduced_graph = DAG(f"{graph.name}_do", remaining_nodes, remaining_edges)

    reduced_parent_order: ParentOrder = {}
    reduced_cpts: CPTMap = {}
    for node in remaining_nodes:
        old_parents = tuple(parent_order[node])
        reduced_parent_order[node] = tuple(
            parent for parent in old_parents if parent not in intervention_nodes
        )
        # For affected child nodes, fix all axes corresponding to intervened parents; CPTs of unaffected nodes are copied unchanged.
        if node in affected:
            index: list[object] = [slice(None)]
            for parent in old_parents:
                if parent in assignments:
                    index.append(STATE_TO_INDEX[parent][assignments[parent]])
                else:
                    index.append(slice(None))
            reduced_cpts[node] = np.asarray(cpts[node][tuple(index)], dtype=float)
        else:
            reduced_cpts[node] = np.array(cpts[node], copy=True)

    validate_cpts(reduced_graph, reduced_cpts, reduced_parent_order)
    return InterventionResult(
        graph=reduced_graph,
        cpts=reduced_cpts,
        parent_order=reduced_parent_order,
        sliced_children=sliced_children,
        assignments=assignments,
    )

assert_frozen_intervention_children()


## 7. Frozen Structure Cache and Audit Tables
When Notebook 00 is loaded, it reruns the four observational FGCR constructions and the four do-FGCR constructions, and compares each parent set, edge count, fill-in count, and parameter dimension against the machine-readable protocol. Notebooks 01–03 therefore consume only verified structures and do not reimplement the graph algorithm independently.

In [7]:
# Encode the edge set as auditable text using the stable node positions.
def canonical_edges(graph: DAG) -> str:
    position = {node: index for index, node in enumerate(graph.nodes)}
    ordered = sorted(graph.edges, key=lambda edge: (position[edge[0]], position[edge[1]]))
    return ";".join(f"{parent}->{child}" for parent, child in ordered)

# Generate a structural hash from the canonical edge text of a graph.
def edge_sha256(graph: DAG) -> str:
    return hashlib.sha256(canonical_edges(graph).encode("utf-8")).hexdigest()

# Ensure that every edge in the output graph is consistent with the specified alpha order.
def assert_order_consistent(graph: DAG, order: Sequence[Node]) -> None:
    position = {node: index for index, node in enumerate(order)}
    if set(position) != set(graph.nodes):
        raise AssertionError("Order node set differs from graph node set")
    if any(position[parent] >= position[child] for parent, child in graph.edges):
        raise AssertionError("Graph contains an edge inconsistent with its requested order")

# Compare the actual and protocol-specified parent sets node by node and report the exact differences.
def _assert_parent_map(
    graph: DAG,
    expected: Mapping[Node, Sequence[Node]],
    label: str,
) -> None:
    actual = parent_order_from_graph(graph)
    if actual != {node: tuple(values) for node, values in expected.items()}:
        differences = {
            node: {"expected": tuple(expected[node]), "actual": actual.get(node)}
            for node in set(expected) | set(actual)
            if tuple(expected.get(node, ())) != tuple(actual.get(node, ()))
        }
        raise AssertionError(f"{label} parent map mismatch: {differences}")

# Rerun the four observational FGCR constructions and verify their parent sets, edge counts, fill-in counts, and parameter dimensions.
def build_observational_structures(
) -> tuple[dict[str, FGCRResult], DAG, pd.DataFrame]:
    remaining = tuple(node for node in NODE_ORDER if node not in CONDITION_NODES)
    G_R = induced_subgraph(ALARM_DAG, remaining, "ALARM_G_R")
    structures: dict[str, FGCRResult] = {}
    rows: list[dict[str, object]] = []
    records = {f"order_{record['id']}": record for record in ORDER_RECORDS}

    for order_id, beta in BETA_ORDERS.items():
        result = fgcr(ALARM_DAG, CONDITION_NODES, beta)
        record = records[order_id]
        _assert_parent_map(result.H_R, EXPECTED_HR_PARENTS[order_id], order_id)
        assert_order_consistent(result.H, result.orders.alpha)
        assert_order_consistent(result.H_R, result.orders.alpha_R)
        if len(result.H_R.edges) != int(record["hr_edge_count"]):
            raise AssertionError(f"{order_id} H_R edge count changed")
        if len(result.added_edges_R) != int(record["fill_edge_count"]):
            raise AssertionError(f"{order_id} fill-edge count changed")
        if parameter_dimension(result.H_R) != int(record["parameter_dimension"]):
            raise AssertionError(f"{order_id} parameter dimension changed")
        structures[order_id] = result
        rows.append({
            "regime": "observational",
            "order_id": order_id,
            "beta": ",".join(beta),
            "alpha_C": ",".join(result.orders.alpha_C),
            "alpha_R": ",".join(result.orders.alpha_R),
            "H_edges": len(result.H.edges),
            "H_R_edges": len(result.H_R.edges),
            "G_R_edges": len(G_R.edges),
            "fill_edges": len(result.added_edges_R),
            "parameter_dimension": parameter_dimension(result.H_R),
            "H_R_sha256": edge_sha256(result.H_R),
            "swaps": result.n_swaps,
            "covered_reversals": result.n_reversals,
            "cover_additions": result.n_cover_additions,
        })
    if len({edge_sha256(item.H_R) for item in structures.values()}) != 4:
        raise AssertionError("The four frozen orders must yield four distinct H_R graphs")
    return structures, G_R, pd.DataFrame(rows)

# Remove nodes only at the graph-structure level for deterministic-do graph construction.
def remove_nodes_from_graph(
    graph: DAG,
    remove: Iterable[Node],
    name: str | None = None,
) -> DAG:
    remove_set = set(remove)
    if not remove_set.issubset(set(graph.nodes)):
        raise ValueError("Cannot remove nodes outside the graph")
    return induced_subgraph(
        graph,
        (node for node in graph.nodes if node not in remove_set),
        name or f"{graph.name}_removed",
    )

# After removing the intervention nodes, rerun the four do-FGCR constructions and verify them against the frozen structures.
def build_interventional_structures(
) -> tuple[DAG, dict[str, FGCRResult], DAG, pd.DataFrame]:
    assert_frozen_intervention_children(ALARM_DAG)
    G_DO = remove_nodes_from_graph(ALARM_DAG, INTERVENTION, "ALARM_do_graph")
    expected_do_parents = {
        node: tuple(values)
        for node, values in FROZEN_PROTOCOL["intervention"]["graph_parents"].items()
    }
    if {
        node: set(G_DO.parents(node)) for node in G_DO.nodes
    } != {
        node: set(values) for node, values in expected_do_parents.items()
    }:
        raise AssertionError("Frozen post-intervention graph changed")
    remaining = tuple(node for node in G_DO.nodes if node not in CONDITION_NODES)
    G_R_DO = induced_subgraph(G_DO, remaining, "ALARM_G_R_do")
    structures: dict[str, FGCRResult] = {}
    rows: list[dict[str, object]] = []
    records = {f"order_{record['id']}": record for record in DO_ORDER_RECORDS}

    for order_id, beta_do in DO_BETA_ORDERS.items():
        result = fgcr(G_DO, CONDITION_NODES, beta_do)
        record = records[order_id]
        _assert_parent_map(result.H_R, EXPECTED_DO_HR_PARENTS[order_id], f"do/{order_id}")
        if len(result.H_R.edges) != int(record["hr_do_edge_count"]):
            raise AssertionError(f"do/{order_id} H_R edge count changed")
        if len(result.added_edges_R) != int(record["fill_edge_count"]):
            raise AssertionError(f"do/{order_id} fill-edge count changed")
        if parameter_dimension(result.H_R) != int(record["parameter_dimension"]):
            raise AssertionError(f"do/{order_id} parameter dimension changed")
        structures[order_id] = result
        rows.append({
            "regime": "do",
            "order_id": order_id,
            "beta": ",".join(beta_do),
            "alpha_C": ",".join(result.orders.alpha_C),
            "alpha_R": ",".join(result.orders.alpha_R),
            "H_edges": len(result.H.edges),
            "H_R_edges": len(result.H_R.edges),
            "G_R_edges": len(G_R_DO.edges),
            "fill_edges": len(result.added_edges_R),
            "parameter_dimension": parameter_dimension(result.H_R),
            "H_R_sha256": edge_sha256(result.H_R),
            "swaps": result.n_swaps,
            "covered_reversals": result.n_reversals,
            "cover_additions": result.n_cover_additions,
        })
    if len({edge_sha256(item.H_R) for item in structures.values()}) != 4:
        raise AssertionError("The four frozen do-orders must yield four distinct H_R graphs")
    return G_DO, structures, G_R_DO, pd.DataFrame(rows)

# Build and verify the four observational H_R graphs at module load time so that Notebooks 01 and 02 can reuse them directly.
OBSERVATIONAL_STRUCTURES, G_R, OBSERVATIONAL_STRUCTURE_AUDIT = (
    build_observational_structures()
)
# Also build the four interventional H_R graphs for direct reuse by Notebook 03.
G_DO, INTERVENTIONAL_STRUCTURES, G_R_DO, INTERVENTIONAL_STRUCTURE_AUDIT = (
    build_interventional_structures()
)
STRUCTURE_AUDIT = pd.concat(
    [OBSERVATIONAL_STRUCTURE_AUDIT, INTERVENTIONAL_STRUCTURE_AUDIT],
    ignore_index=True,
)
# Include the edge hashes of all eight H_R graphs in the identities of subsequent experiment protocols.
STRUCTURE_HASHES = {
    f"obs_{order_id}": edge_sha256(result.H_R)
    for order_id, result in OBSERVATIONAL_STRUCTURES.items()
} | {
    f"do_{order_id}": edge_sha256(result.H_R)
    for order_id, result in INTERVENTIONAL_STRUCTURES.items()
}


## 8. Independent Self-Checks in Notebook 00
The self-checks cover the frozen protocol header, the four observational and interventional structures, the d-separation conditions for minimal I-maps, generic multi-intervention slicing, parameter-bank round trips, and the three exact probability chains based on the oracle, VE, and FGCR. The audit workbook is written only after all checks pass.

In [8]:
# Compute the ancestral closure required for reverse-search d-separation tests.
def _ancestors_for_dsep(graph: DAG, nodes: Iterable[Node]) -> set[Node]:
    result = set(nodes)
    stack = list(nodes)
    while stack:
        node = stack.pop()
        for parent in graph.parents(node):
            if parent not in result:
                result.add(parent)
                stack.append(parent)
    return result

# Determine d-separation by undirected connectivity after removing the conditioning set from the ancestral moral graph.
def d_separated(
    graph: DAG,
    X: Iterable[Node],
    Y: Iterable[Node],
    Z: Iterable[Node],
) -> bool:
    X, Y, Z = set(X), set(Y), set(Z)
    if (X & Y) or (X & Z) or (Y & Z):
        raise ValueError("X, Y and Z must be pairwise disjoint")
    ancestors = _ancestors_for_dsep(graph, X | Y | Z)
    adjacency = {node: set() for node in ancestors}
    for parent, child in graph.edges:
        if parent in ancestors and child in ancestors:
            adjacency[parent].add(child)
            adjacency[child].add(parent)
    for child in ancestors:
        parents = [parent for parent in graph.parents(child) if parent in ancestors]
        for left, right in combinations(parents, 2):
            adjacency[left].add(right)
            adjacency[right].add(left)
    active = ancestors - Z
    frontier = [node for node in X if node in active]
    visited = set(frontier)
    while frontier:
        node = frontier.pop()
        if node in Y:
            return False
        for neighbor in adjacency[node]:
            if neighbor in active and neighbor not in visited:
                visited.add(neighbor)
                frontier.append(neighbor)
    return True

#  Verify the I-map and minimality properties of H and H_R using the recursive basis and edge-by-edge deletion checks.
def verify_fgcr_mdi_properties(
    source_graph: DAG,
    conditioning_nodes: Iterable[Node],
    result: FGCRResult,
) -> None:
    condition_set = set(conditioning_nodes)
    for index, node in enumerate(result.orders.alpha):
        predecessors = set(result.orders.alpha[:index])
        parents = set(result.H.parents(node))
        nonparents = predecessors - parents
        if nonparents and not d_separated(source_graph, {node}, nonparents, parents):
            raise AssertionError(f"H is not an I-map at {node}")
        for parent in parents:
            if d_separated(source_graph, {node}, {parent}, parents - {parent}):
                raise AssertionError(f"H edge {parent}->{node} is redundant")
    for index, node in enumerate(result.orders.alpha_R):
        predecessors = set(result.orders.alpha_R[:index])
        parents = set(result.H_R.parents(node))
        nonparents = predecessors - parents
        if nonparents and not d_separated(
            source_graph,
            {node},
            nonparents,
            parents | condition_set,
        ):
            raise AssertionError(f"H_R is not an I-map of I(G)^C at {node}")
        for parent in parents:
            if d_separated(
                source_graph,
                {node},
                {parent},
                (parents - {parent}) | condition_set,
            ):
                raise AssertionError(f"H_R edge {parent}->{node} is redundant under C")

# Run the complete regression suite for structures, probabilities, the parameter bank, and generic interventions, and write the audit tables.
def run_fgcr_core_selfcheck(
    project_root: Path = PROJECT_ROOT,
    exact_tolerance: float = 1e-10,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    import tempfile

    tests: list[dict[str, object]] = []
    probability_rows: list[dict[str, object]] = []

    # Execute hard assertions at module load time so that protocol drift is exposed before any experiment begins.
    validate_frozen_header()
    assert_frozen_intervention_children()
    tests.append({
        "test": "frozen_header_node_order_kahn_do_beta_and_children",
        "passed": True,
    })

    for result in OBSERVATIONAL_STRUCTURES.values():
        verify_fgcr_mdi_properties(ALARM_DAG, CONDITION_NODES, result)
    for result in INTERVENTIONAL_STRUCTURES.values():
        verify_fgcr_mdi_properties(G_DO, CONDITION_NODES, result)
    tests.append({
        "test": "eight_fgcr_graphs_are_minimal_imaps_by_dseparation",
        "passed": True,
        "cases": 8,
    })

    rng = np.random.default_rng(20260825)
    random_cases = 0
    for n_nodes in range(3, 8):
        nodes = tuple(f"v{index}" for index in range(n_nodes))
        for _ in range(30):
            edges = frozenset(
                (nodes[left], nodes[right])
                for left in range(n_nodes)
                for right in range(left + 1, n_nodes)
                if rng.random() < 0.28
            )
            graph = DAG("random_selfcheck", nodes, edges)
            condition_size = int(rng.integers(1, n_nodes))
            condition = set(rng.choice(nodes, size=condition_size, replace=False).tolist())
            result = fgcr(graph, condition, nodes)
            verify_fgcr_mdi_properties(graph, condition, result)
            random_cases += 1
    tests.append({
        "test": "random_small_dag_direct_dseparation",
        "passed": True,
        "cases": random_cases,
    })

    cpts = generate_random_cpts(0)
    repeated = generate_random_cpts(0)
    if cpts_sha256(cpts, PARENT_ORDER) != cpts_sha256(repeated, PARENT_ORDER):
        raise AssertionError("Parameter generation is not reproducible")
    tests.append({"test": "pcg64_seedsequence_parameter_reproducibility", "passed": True})

    with tempfile.TemporaryDirectory(prefix="alarm_00_bank_") as temporary:
        path = Path(temporary) / "parameter_bank_roundtrip.xlsx"
        small_bank = generate_parameter_bank((0, 1))
        write_parameter_bank(path, small_bank)
        restored_bank = load_parameter_bank(path)
        if restored_bank.content_sha256 != small_bank.content_sha256:
            raise AssertionError("Parameter bank round trip changed its logical content")
    tests.append({"test": "multistate_parameter_bank_excel_roundtrip", "passed": True})

    # First generate the observational ground truth using independent enumeration, then verify VE and the four FGCR paths.
    truth_oracle = ancestral_enumeration_query(
        cpts,
        ALARM_DAG,
        PARENT_ORDER,
        TARGET,
        CONDITION,
    )
    truth_ve = ve_query(cpts, PARENT_ORDER, TARGET, CONDITION)
    oracle_errors = distribution_errors(truth_oracle, truth_ve)
    probability_rows.append({
        "regime": "observational",
        "method": "source_ve_vs_independent_oracle",
        **oracle_errors,
    })
    if max(oracle_errors.values()) > exact_tolerance:
        raise AssertionError("VE and independent enumeration oracle disagree")

    ic_cpts, ic_parent_order = compile_conditional_cpts(cpts, PARENT_ORDER, G_R, CONDITION)
    ic_distribution = ve_query(ic_cpts, ic_parent_order, TARGET)
    ic_errors = distribution_errors(truth_oracle, ic_distribution)
    probability_rows.append({
        "regime": "observational",
        "method": "induced_graph_control",
        **ic_errors,
    })
    if ic_errors["tvd"] <= exact_tolerance:
        raise AssertionError("Fixed parameter 0 makes the induced-graph control trivial")

    for order_id, result in OBSERVATIONAL_STRUCTURES.items():
        compiled, compiled_parent_order = compile_conditional_cpts(
            cpts,
            PARENT_ORDER,
            result.H_R,
            CONDITION,
        )
        distribution = ve_query(compiled, compiled_parent_order, TARGET)
        errors = distribution_errors(truth_oracle, distribution)
        probability_rows.append({
            "regime": "observational",
            "method": f"fgcr_{order_id}",
            **errors,
        })
        if max(errors.values()) > exact_tolerance:
            raise AssertionError(f"Exact FGCR factorization failed for {order_id}")

    do_model = intervene_and_remove(ALARM_DAG, cpts, PARENT_ORDER, INTERVENTION)
    if do_model.sliced_children != EXPECTED_INTERVENTION_CHILDREN:
        raise AssertionError("Generic intervention did not report the frozen child set")
     # Compute a separate independent oracle for the interventional model rather than reusing the observational ground truth.
    truth_do_oracle = ancestral_enumeration_query(
        do_model.cpts,
        do_model.graph,
        do_model.parent_order,
        TARGET,
        CONDITION,
    )
    truth_do_ve = ve_query(do_model.cpts, do_model.parent_order, TARGET, CONDITION)
    do_oracle_errors = distribution_errors(truth_do_oracle, truth_do_ve)
    probability_rows.append({
        "regime": "do",
        "method": "source_ve_vs_independent_oracle",
        **do_oracle_errors,
    })
    if max(do_oracle_errors.values()) > exact_tolerance:
        raise AssertionError("do-VE and independent enumeration oracle disagree")

    for order_id, result in INTERVENTIONAL_STRUCTURES.items():
        compiled, compiled_parent_order = compile_conditional_cpts(
            do_model.cpts,
            do_model.parent_order,
            result.H_R,
            CONDITION,
        )
        distribution = ve_query(compiled, compiled_parent_order, TARGET)
        errors = distribution_errors(truth_do_oracle, distribution)
        probability_rows.append({
            "regime": "do",
            "method": f"fgcr_{order_id}",
            **errors,
        })
        if max(errors.values()) > exact_tolerance:
            raise AssertionError(f"Exact do-FGCR factorization failed for {order_id}")

    # Use two common-parent interventions to explicitly verify that the generic function can remove multiple parent axes simultaneously.
    multi = intervene_and_remove(
        ALARM_DAG,
        cpts,
        PARENT_ORDER,
        {"INT": "NORMAL", "KINK": "TRUE"},
    )
    expected_vlng = cpts["VLNG"][:, STATE_TO_INDEX["INT"]["NORMAL"], STATE_TO_INDEX["KINK"]["TRUE"], :]
    expected_prss = cpts["PRSS"][:, STATE_TO_INDEX["INT"]["NORMAL"], STATE_TO_INDEX["KINK"]["TRUE"], :]
    if multi.parent_order["VLNG"] != ("VTUB",) or multi.parent_order["PRSS"] != ("VTUB",):
        raise AssertionError("Multi-intervention failed to remove both parent axes")
    if not np.array_equal(multi.cpts["VLNG"], expected_vlng):
        raise AssertionError("VLNG multi-parent slice is incorrect")
    if not np.array_equal(multi.cpts["PRSS"], expected_prss):
        raise AssertionError("PRSS multi-parent slice is incorrect")
    tests.append({"test": "generic_multi_intervention_child_slicing", "passed": True})

    probability_frame = pd.DataFrame(probability_rows)
    max_exact = float(
        probability_frame.loc[
            probability_frame["method"] != "induced_graph_control",
            ["tvd", "max_abs_error"],
        ].to_numpy().max()
    )
    tests.append({
        "test": "oracle_ve_and_eight_fgcr_probability_paths",
        "passed": max_exact <= exact_tolerance,
        "max_error": max_exact,
    })

    test_frame = pd.DataFrame(tests)
    output = Path(project_root) / "result" / "00_fgcr_common_core_selfcheck.xlsx"
    write_excel_atomic(output, {
        "unit_tests": test_frame,
        "structures": STRUCTURE_AUDIT,
        "probability": probability_frame,
        "environment": pd.DataFrame([environment_record()]),
    })
    return test_frame, STRUCTURE_AUDIT.copy(), probability_frame


In [9]:
# This cell prints the results and retains the audit workbook only after all self-checks have passed.
SELF_CHECK_TESTS, SELF_CHECK_STRUCTURES, SELF_CHECK_PROBABILITY = (
    run_fgcr_core_selfcheck()
)
print(SELF_CHECK_TESTS.to_string(index=False))
print("\nMaximum exact-path error:", SELF_CHECK_PROBABILITY.loc[
    SELF_CHECK_PROBABILITY["method"] != "induced_graph_control",
    ["tvd", "max_abs_error"],
].to_numpy().max())


                                              test  passed  cases    max_error
frozen_header_node_order_kahn_do_beta_and_children    True    NaN          NaN
eight_fgcr_graphs_are_minimal_imaps_by_dseparation    True    8.0          NaN
               random_small_dag_direct_dseparation    True  150.0          NaN
      pcg64_seedsequence_parameter_reproducibility    True    NaN          NaN
         multistate_parameter_bank_excel_roundtrip    True    NaN          NaN
          generic_multi_intervention_child_slicing    True    NaN          NaN
        oracle_ve_and_eight_fgcr_probability_paths    True    NaN 3.894454e-16

Maximum exact-path error: 3.894454203567932e-16
